<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Deep_neural_network_of_L_layers/blob/main/Copy_of_L_layered_Deep_learning_from_scrach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [158]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles

X, Y = make_circles(n_samples=6000,noise=0.15,factor=0.5)

In [159]:

print(Y.shape[0] , X.T.shape)


6000 (2, 6000)


In [160]:
print(X.shape[1])

2


In [161]:
def init_parameters(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * 0.01
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b


In [162]:
def init_parameters_he(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * np.sqrt(2/layers_dims[l-1])
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b

In [163]:
def init_parameters_random(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * 0.01
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b


In [164]:
def init_parameters_zeros(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.zeros((layers_dims[l] , layers_dims[l-1]))
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b

In [165]:
def forward(W , b , X , L):
  Zs = []
  As = []
  A = X.T
  As.append(A)

  def relu(Z):
    return np.maximum(0 , Z)
  def sigmoid(Z):
    return 1/(1 + np.exp(-Z))

  for l in range(1 , L-1):
    A_prev = A
    Z = np.dot(W[l],A_prev)+b[l]
    Zs.append(Z)
    A = relu(Z)
    As.append(A)

  ZL = np.dot(W[L-1],A)+b[L-1]
  AL = sigmoid(ZL)
  Zs.append(ZL)
  As.append(AL)
  return Zs , As

In [166]:
def cost(Y , Y_hat):
  m = Y.shape[1]
  temp = Y * np.log(Y_hat) + (1-Y) * np.log(1-Y_hat)
  return -np.sum(temp)/m

In [167]:
def backward(W, Zs, As, Y, L):

    dW = {}

    db = {}

    m = Y.shape[1]

    AL = As[-1]

    dZ = AL - Y

    for l in reversed(range(1, L)):
        dW[l] = (1/m) * np.dot(dZ, As[l-1].T)
        db[l] = (1/m) * np.sum(dZ,axis=1,keepdims=True)
        if l > 1:
            dA = np.dot(W[l].T, dZ)
            dZ = dA * (Zs[l-2] > 0)

    return dW, db

In [168]:
def backward_with_L2(W, Zs, As, Y, L , lambd):

    dW = {}

    db = {}

    m = Y.shape[1]

    AL = As[-1]

    dZ = AL - Y

    for l in reversed(range(1, L)):
        dW[l] = (1/m) * np.dot(dZ, As[l-1].T) + (lambd/m)*W[l]
        db[l] = (1/m) * np.sum(dZ,axis=1,keepdims=True)
        if l > 1:
            dA = np.dot(W[l].T, dZ)
            dZ = dA * (Zs[l-2] > 0)

    return dW, db

In [169]:
def gradient(W , b , X , Y , L , alpha):
  Zs , As = forward(W , b , X , L)
  dW , db = backward(W , Zs , As , Y , L)
  for l in range(1, L):
    W[l] = W[l] - alpha * dW[l]
    b[l] = b[l] - alpha * db[l]
  return W , b

In [170]:
def gradient_with_L2(W , b , X , Y , L , alpha , lambd):
  Zs , As = forward(W , b , X , L)
  dW , db = backward_with_L2(W , Zs , As , Y , L , lambd)
  for l in range(1, L):
    W[l] = W[l] - alpha * dW[l]
    b[l] = b[l] - alpha * db[l]
  return W , b

In [171]:
def L2_cost(W, b, X, Y, L, lambd):

    m = X.shape[1]

    L2 = 0

    for l in range(1, L):
        L2 += np.sum(W[l]**2)

    _, As = forward(W, b, X, L)
    cross = cost(Y, As[-1])
    reg = (lambd/(2*m))*L2

    # print("Cross Entropy:", cross)
    # print("L2 Penalty:", reg)

    # return cross + reg

    return cost(Y, As[-1]) + (lambd/(2*m))*L2

In [172]:
def model(dim , X , Y , initialization , reg , alpha):

  if initialization == "zeros":
        W , b = init_parameters_zeros(dim)
  elif initialization == "random":
        W , b = init_parameters_random(dim)
  elif initialization == "he":
        W , b = init_parameters_he(dim)

  # W , b = init_parameters(dim)
  L = len(dim)

  for i in range(10000):
    if(reg == "L2"):
      W , b = gradient_with_L2(W , b , X , Y , L , alpha , 0.7)
    elif(reg == "normal"):
      W , b = gradient(W , b , X , Y , L , alpha)
    if i % 1000 == 0:
        _, As = forward(W, b, X, L)
        if(reg == "L2"):
          c = L2_cost(W, b, X, Y, L, 0.7)
        elif(reg == "normal"):
          c = cost(Y, As[-1])
        print(f"Iteration {i}: Cost = {c:.4f}")
  return W , b

In [173]:
# Y = Y.reshape(1, -1)
# W , b = model([2 , 3  , 1] , X , Y , initialization="he" , reg= "L2" , alpha = 0.01)


In [174]:
# L = len([2 , 3 , 1])
# Zs, As = forward(W, b, X, L)
# predictions = (As[-1] >= 0.5)
# accuracy = np.mean(predictions == Y) * 100
# print(f"Accuracy: {accuracy:.2f}%")

optimization techniques


In [175]:
print(X.shape[0])
print(Y.shape)

6000
(6000,)


In [176]:
epochs = 1000
batch_size = 64
dim = [2 , 3 , 1]
L = len(dim)
W, b = init_parameters_he(dim)
if Y.ndim == 1:
   Y = Y.reshape(1, -1)

def random_mini_batches(X, Y, batch_size=64, seed=None):

    if seed is not None:
        np.random.seed(seed)

    m = X.shape[0]
    mini_batches = []

    permutation = np.random.permutation(m)

    X_shuffled = X[permutation, :]
    Y_shuffled = Y[:, permutation]

    num_complete_batches = m // batch_size

    for k in range(num_complete_batches):
        start = k * batch_size
        end = start + batch_size

        X_batch = X_shuffled[start:end, :]
        Y_batch = Y_shuffled[:, start:end]
        mini_batches.append((X_batch, Y_batch))


    if m % batch_size != 0:
        start = num_complete_batches * batch_size
        X_batch = X_shuffled[start:m, :]
        Y_batch = Y_shuffled[:, start:m]
        mini_batches.append((X_batch, Y_batch))

    return mini_batches

for epoch in range(epochs):

    mini_batches = random_mini_batches(X, Y, batch_size)

    epoch_cost = 0

    for X_batch, Y_batch in mini_batches:

        W , b = gradient(W , b , X_batch , Y_batch , L , 0.01)

        _, As_batch = forward(W, b, X_batch, L)
        batch_cost = cost(Y_batch, As_batch[-1])
        epoch_cost += batch_cost

    epoch_cost /= len(mini_batches)

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Cost = {epoch_cost:.5f}")

Epoch 0: Cost = 0.68021
Epoch 100: Cost = 0.29407
Epoch 200: Cost = 0.21602
Epoch 300: Cost = 0.18806
Epoch 400: Cost = 0.17332
Epoch 500: Cost = 0.16476
Epoch 600: Cost = 0.15845
Epoch 700: Cost = 0.15429
Epoch 800: Cost = 0.15125
Epoch 900: Cost = 0.14870


In [177]:
L = len([2 , 3 , 1])
Zs, As = forward(W, b, X, L)
predictions = (As[-1] >= 0.5)
accuracy = np.mean(predictions == Y) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 94.60%
